In [ ]:
import os
from openai import OpenAI
import pandas as pd
import numpy as np
import re
from pypinyin import lazy_pinyin
from rapidfuzz import fuzz
import math
from uuid import uuid4 as uuid
from dotenv import load_dotenv
import subprocess
from tqdm import tqdm
import json
load_dotenv(".env")

root = "/mnt/NextcloudSacmData/sacm.av/files/Recordings"
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
# df = pd.read_pickle("song_embeddings_large.pkl")
# embeddings = np.vstack(df["embedding"].values).astype(np.float32)
# embeddings /= np.linalg.norm(embeddings, axis=1, keepdims=True)
df = pd.read_csv("songs.csv")
df.sample()

,code,type,title,lyrics,pinyin
242,Y5-7,PnW,永远不分离,我要像一颗树按时结果，栽在溪水旁，喜爱祢的话语，昼夜思想。\n这是有福的。祢就像葡萄树，我是...,wo yao xiang yi ke shu an shi jie guo zai zai ...


In [ ]:
def get_embedding(text):
    text = re.sub(r"[，。！？、“”：；\n]", " ", text)
    response = client.embeddings.create(
        model="text-embedding-3-large",
        input=text
    )
    return response.data[0].embedding


def pinyin(text):
    text = re.sub(r"[，。！？、“”：；\n]", " ", text)
    result = " ".join(lazy_pinyin(text))
    return re.sub(r"\s+", " ", result).strip()


def log_score(x, k=0.1):
    return math.log(1 + k*x) / math.log(1 + 100*k)


def windows(tokens, size, step):
    if len(tokens) <= size:
        yield " ".join(tokens)
    else:
        for i in range(0, len(tokens) - size + 1, step):
            yield " ".join(tokens[i:i+size])


def best_window_score(query_py, lyrics_py, size=50, step=10):
    query_tokens = query_py.split()
    lyric_tokens = lyrics_py.split()
    score = max(
        fuzz.ratio(qw, lw)
        for qw in windows(query_tokens, size, step)
        for lw in windows(lyric_tokens, size, step)
    )
    return score / 100


def weighted_avg(a, b, alpha=0.5, beta=0.5):
    return (a * alpha + b * beta) / 2


def get_duration(filepath):
    duration = float(subprocess.check_output([
        "ffprobe",
        "-v", "error",
        "-show_entries", "format=duration",
        "-of", "default=noprint_wrappers=1:nokey=1",
        filepath,
    ]).decode().strip())
    return duration


def split_to_limit(filepath, limit=26_214_400, margin=0.90, out_dir="tmp"):
    size = os.path.getsize(filepath)
    if size <= limit:
        return [filepath]
    os.makedirs(out_dir, exist_ok=True)
    duration = get_duration(filepath)
    bitrate_kbps = 128
    chunk_seconds = max(1, int(limit * margin * 8 / (bitrate_kbps * 1000)))
    chunk_paths = []
    
    for start in range(0, math.ceil(duration), chunk_seconds):
        chunk_path = os.path.join(out_dir, f"{uuid()}.mp3")
        subprocess.run([
            "ffmpeg",
            "-y",
            "-ss", str(start),
            "-t", str(chunk_seconds),
            "-i", filepath,
            "-vn",
            "-c:a", "libmp3lame",
            "-b:a", f"{bitrate_kbps}k",
            chunk_path,
        ], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        chunk_paths.append(chunk_path)
        print(
            f"chunk {len(chunk_paths)}: "
            f"{os.path.getsize(chunk_path):,} bytes "
            f"(limit {limit:,}) -> {chunk_path}"
        )
    return chunk_paths

In [75]:
def match_zoom_to_sq(d, tol=10, verbose=False):
    files = sorted(os.listdir(f"{root}/{d}"))
    zoom_files = [f for f in files if f.startswith("ZOOM")]
    sq_files = [f for f in files if not f.startswith("ZOOM")]
    
    if not zoom_files or not sq_files:
        return {}

    durations = {f: int(get_duration(f"{root}/{d}/{f}")) for f in files}
    if verbose:
        print(json.dumps(durations, indent=2, ensure_ascii=False))

    reduced_sq_files = []
    for f in sq_files:
        other_files = [x for x in reduced_sq_files if not x.startswith("SQ")]
        if f.startswith("SQ") or not any(durations[f] == durations[x] for x in other_files):
            reduced_sq_files.append(f)
    sq_files = reduced_sq_files

    if verbose:
        print(zoom_files)
        print(sq_files)

    # If same number of files, just pair in order
    if len(zoom_files) == len(sq_files):
        ordered_matches = dict(zip(zoom_files, sq_files))
        # Make sure the sizes match though
        if all(abs(durations[z] - durations[s]) < tol for z, s in ordered_matches.items()):
            return ordered_matches

    # Otherwise, take the largest filesize (P&W) pair as reference
    matches = {}

    def traverse(ref_zoom_idx, ref_sq_idx, zoom_list, sq_list):
        z_idx = ref_zoom_idx + 1
        s_idx = ref_sq_idx + 1
        while z_idx < len(zoom_list) and s_idx < len(sq_list):
            zoom_file = zoom_list[z_idx]
            for i, sq_file in enumerate(sq_list):
                if sq_file.startswith("SQ") and i < s_idx:
                    continue
                if verbose:
                    print(f"{i=} {s_idx=} {zoom_file} ({durations[zoom_file]}) : {sq_file} ({durations[sq_file]})")
                if abs(durations[zoom_file] - durations[sq_file]) > tol:
                    continue
                matches[zoom_file] = sq_file
                if sq_file.startswith("SQ"):
                    s_idx = i + 1
                break
            z_idx += 1

    def prune_same_values(d):
        counts = {v: len([k for k in d if d[k] == v]) for v in d.values()}
        return {k: v for k, v in d.items() if counts[v] == 1}

    largest_zoom_file = max(zoom_files, key=lambda f: durations[f])
    largest_sq_file = max(sq_files, key=lambda f: durations[f])
    if not largest_sq_file.startswith("SQ"):
        # since order can't be gleaned from filename, just traverse the whole list
        traverse(-1, -1, zoom_files, sq_files)
        return prune_same_values(matches)

    if abs(durations[largest_zoom_file] - durations[largest_sq_file]) <= tol:
        matches[largest_zoom_file] = largest_sq_file
        zoom_idx = zoom_files.index(largest_zoom_file)
        sq_idx = sq_files.index(largest_sq_file)
    else:
        cost = np.array([
            [abs(durations[z] - durations[s]) for s in sq_files]
            for z in zoom_files
        ])
        zoom_idx, sq_idx = np.unravel_index(np.argmin(cost), cost.shape)
        matches[zoom_files[zoom_idx]] = sq_files[sq_idx]

    traverse(zoom_idx, sq_idx, zoom_files, sq_files)
    traverse(len(zoom_files) - zoom_idx - 1, len(sq_files) - sq_idx - 1, zoom_files[::-1], sq_files[::-1])

    return prune_same_values(matches)

matches = match_zoom_to_sq("2026-04-11")
print(json.dumps(matches, indent=2, ensure_ascii=False))

{
  "ZOOM0306_你真伟大.mp3": "SQ-ST306_你真伟大.mp3",
  "ZOOM0307_救主耶稣万福恩源.mp3": "SQ-ST307_救主耶稣万福恩源.mp3",
  "ZOOM0312_点燃.mp3": "SQ-ST309_点燃.mp3"
}


In [78]:
for d in sorted(os.listdir(root), reverse=True):
    if not d.startswith("2026"):
        continue
    files = sorted(os.listdir(f"{root}/{d}"))
    for f in files:
        if bool(re.search(r'[\u4e00-\u9fff]', f)):
            continue
        filepath = f"{root}/{d}/{f}"
        duration = get_duration(filepath)
        mins, secs = int(duration // 60), int(duration % 60)
        print(f"{filepath} {mins:02d}:{secs:02d}")

/mnt/NextcloudSacmData/sacm.av/files/Recordings/2026-06-14/SQ-ST388.mp3 00:32
/mnt/NextcloudSacmData/sacm.av/files/Recordings/2026-04-19/SQ-ST323.mp3 01:47
/mnt/NextcloudSacmData/sacm.av/files/Recordings/2026-04-19/ZOOM0326.mp3 01:43
/mnt/NextcloudSacmData/sacm.av/files/Recordings/2026-03-08/ZOOM0271.mp3 00:12
/mnt/NextcloudSacmData/sacm.av/files/Recordings/2026-03-07/SQ-ST262.mp3 00:10
/mnt/NextcloudSacmData/sacm.av/files/Recordings/2026-03-07/ZOOM0258.mp3 00:12
/mnt/NextcloudSacmData/sacm.av/files/Recordings/2026-02-28/SQ-ST251.mp3 00:11
/mnt/NextcloudSacmData/sacm.av/files/Recordings/2026-02-28/ZOOM0248.mp3 00:11


In [19]:
print(split_to_limit(f"{root}/2026-05-08 (祷告会)/SQ-ST346.mp3"))

chunk 1: 23,585,062 bytes (limit 26,214,400) -> tmp/b76a32bb-97fc-499c-9cf6-2ce0c8f9af7f.mp3
chunk 2: 23,585,062 bytes (limit 26,214,400) -> tmp/35b7d494-22fc-4bff-91c0-113629c84c61.mp3
chunk 3: 23,585,062 bytes (limit 26,214,400) -> tmp/eaaceb3c-0d40-4fd6-b087-24c4e3342857.mp3
chunk 4: 23,344,318 bytes (limit 26,214,400) -> tmp/9a0d72a0-bbe4-46e5-a8ac-4e17bfcf9cba.mp3
['tmp/b76a32bb-97fc-499c-9cf6-2ce0c8f9af7f.mp3', 'tmp/35b7d494-22fc-4bff-91c0-113629c84c61.mp3', 'tmp/eaaceb3c-0d40-4fd6-b087-24c4e3342857.mp3', 'tmp/9a0d72a0-bbe4-46e5-a8ac-4e17bfcf9cba.mp3']


In [62]:
# filepath = f"{root}/2026-01-24/SQ-ST196.mp3"
lyrics = ""
for filepath in tqdm([f"{root}/2026-04-11/ZOOM0311_点燃.mp3"]):
    audio_file = open(filepath, "rb")
    transcription = client.audio.transcriptions.create(
        # model="gpt-4o-transcribe", 
        model="whisper-1", 
        file=audio_file,
        language="zh",
    )
    lyrics += transcription.text
lyrics

100%|██████████| 1/1 [00:18<00:00, 18.11s/it]


'我們不再懼怕 無論明天如何 我們都能面對 親愛的主耶穌 我們來到祢面前 求祢不要讓我們只停留在這裡 只停留在忍受這份苦難 主啊 我們也邀請祢更信我們 更信我們 更信我們 更信我們 更信我們 更信我們 主啊我們也邀請祢更信我們的生命 帶領我們 更靠近我們 也讓我們的生命越來越香 弟兄姊妹 當我們唱這首歌的時候 讓我們帶著禱告的信 求主耶穌來更信 來改變我們的生命 幾十光年的路 努力淹沒我的心 我要向你舉行 奮鬥尋求你的信 求見你我貼近你 無論到處地球 渴望能重新等你 聽到你的謊言 讓背景都再次安靜 成為分享的生命 求你點燃 點燃我們記住唯一 火熱的心 祝你的海天能 奔向年輕我們 向我們獻殷勤 求你點燃 興起我們生命成為 新鮮的火氣 祝你出現大腦 一直釋放我們 求你再次證明 點燃我的心 無論到處地球 渴望能重新等你 聽到你的謊言 讓背景都再次安靜 成為分享的生命 求你點燃 點燃我們記住唯一 火熱的心 祝你的海天能 奔向年輕我們 向我們獻殷勤 求你點燃 興起我們生命成為 新鮮的火氣 祝你出現大腦 一直釋放我們 求你再次證明 點燃我的心 如同吹起陽光的星星 護著你讓我再次燃起 愛曾經卻被獻上 自己生命中的記憶 如同吹起陽光的星星 護著你讓我再次燃起 愛曾經卻被獻上 自己生命中的記憶 求你點燃 點燃我們記住唯一 火熱的心 祝你的海天能 奔向年輕我們 向我們獻殷勤 求你點燃 興起我們生命成為 新鮮的火氣 祝你出現大腦 一直釋放我們 求你再次證明 點燃我的心 祝你出現大腦 一直釋放我們 求你再次證明 點燃我的心'

In [63]:
titles = {}
title_to_last_chunk_idx = {}

query_lyrics = re.sub(r"[，。！、\n]", " ", lyrics)

chunk_size = 120
for i, start in enumerate(range(0, len(query_lyrics), chunk_size)):
    chunk = query_lyrics[start:start + chunk_size]
    if len(chunk) < 50:
        continue
    # query_embedding = get_embedding(chunk)
    # query_embedding /= np.linalg.norm(query_embedding)
    # scores = embeddings @ query_embedding
    # alpha, beta = (0.1, 0.9) if len(chunk) < chunk_size / 2 else (0.3, 0.7)
    # scores = [weighted_avg(
    #     score, best_window_score(pinyin(chunk), df.pinyin[i], size=len(chunk)//2, step=5), alpha, beta,
    # ) for i, score in enumerate(scores)]
    query_py = pinyin(chunk)
    scores = [best_window_score(query_py, lyric_py, size=min(len(chunk), 100), step=5) for lyric_py in df.pinyin]
    best_idx = np.argmax(scores)
    best_title = df.iloc[best_idx]["title"]
    best_score = scores[best_idx]
    print(f"[{start}:{start+chunk_size}] {best_title=}, {best_score=}")
    if best_title in titles and (i - title_to_last_chunk_idx.get(best_title, -5)) <= 2:
        titles[best_title] = max(titles[best_title], best_score) * 1.2
    else:
        titles[best_title] = best_score
    title_to_last_chunk_idx[best_title] = i

print(f"{titles=}")
final_titles = [title for title, score in titles.items() if score > 0.7]
print(f"Songs: {'_'.join(final_titles)}")

[0:120] best_title='爱使我们勇敢+我们爱', best_score=0.5807259073842304
[120:240] best_title='点燃', best_score=0.5707196029776676
[240:360] best_title='点燃', best_score=0.8386308068459658
[360:480] best_title='点燃', best_score=0.8602941176470589
[480:600] best_title='点燃', best_score=0.6951066499372647
[600:720] best_title='点燃', best_score=0.7045454545454546
titles={'爱使我们勇敢+我们爱': 0.5807259073842304, '点燃': 1.7389848410757944}
Songs: 点燃


In [79]:
print(" ".join(df.loc[df.title == "在基督里有平安"].lyrics.to_list()))

在世界上，我们有苦难；在基督里，我们有平安。
愿我口不出埋怨，愿我手不行恶端，愿我一生单单颂赞。
我不求外在困难短减，我只求内心平安加添，因我深知有永生恩典，等候在我的面前。
我不求外在困难短减，我只求内心平安加添，因我深信有生命冠冕，在耶稣基督里面。


In [ ]:
chunk = query_lyrics[2640:2760]
print("Query:", chunk)
query_pinyin = pinyin(chunk)
query_embedding = get_embedding(chunk)
query_embedding /= np.linalg.norm(query_embedding)
for t in ["宁静谷"]:
    inds = df.loc[df.title == t].index
    for idx in inds:
        print(f"[{idx}] {t}: {df.pinyin[idx]}")
        fuzz_score = best_window_score(query_pinyin, df.pinyin[idx], size=100, step=3)
        print(f"{fuzz_score}")

Query:  我学会了信靠他 依靠他 有一次当我 向一位朋友 倾诉我的挣扎时 他推荐我 他推荐给我一首 藏民之群的歌 叫《宁静谷》 歌词中写道 生活中的仓促 生命里的难处 只愿向他来倾诉 平安祝福在这谷 我觉得这首歌 正好讲述了 那段时期 上帝如何 把
[77] 宁静谷: zai wo xin ling shen chu you yi zuo ning jing gu wo he wo qin ai de zhu zai qi zhong an ran man bu sheng huo zhong de cang cu sheng ming li de nan chu zhi yuan xiang ta lai qing su ping an zhu fu zai zhe gu wo yu wo zhu xiang yue zhi chu chang yang zhe fen ning jing an xiang jiu xiang shi zai tian tang wo yu wo zhu xiang yue zhi chu zhu ling wo guo si yin you gu shi wo xi le zou ren sheng lu
score=0.5467158003484595, fuzz_score=0.5852417302798982, 0.2868419756502333


In [ ]:
def get_titles(filepath):
    print(f"Processing {filepath}")

    cropped_paths = split_to_limit(filepath)
    lyrics = ""
    for filepath in tqdm(cropped_paths):
        audio_file = open(filepath, "rb")
        transcription = client.audio.transcriptions.create(
            # model="gpt-4o-transcribe", 
            model="whisper-1", 
            file=audio_file,
            language="zh",
        )
        lyrics += transcription.text

    if not lyrics:
        return []

    titles = {}
    title_to_last_chunk_idx = {}

    query_lyrics = re.sub(r"[，。！、\n]", " ", lyrics)

    chunk_size = 120
    for i, start in enumerate(range(0, len(query_lyrics), chunk_size)):
        chunk = query_lyrics[start:start + chunk_size]
        if len(chunk) < 50:
            continue
        query_py = pinyin(chunk)
        scores = [best_window_score(query_py, lyric_py, size=min(len(chunk), 100), step=5) for lyric_py in df.pinyin]
        best_idx = np.argmax(scores)
        best_title = df.iloc[best_idx]["title"]
        best_score = scores[best_idx]
        print(f"[{start}:{start+chunk_size}] {best_title=}, {best_score=}")
        if best_title in titles and (i - title_to_last_chunk_idx.get(best_title, -5)) <= 2:
            titles[best_title] = max(titles[best_title], best_score) * 1.2
        else:
            titles[best_title] = best_score
        title_to_last_chunk_idx[best_title] = i

    print(f"{titles=}")
    duration = get_duration(filepath)
    if duration > 3 * 60:
        final_titles = [title for title, score in titles.items() if score > 0.7]
    else:
        best_title = max(titles, key=titles.get)
        final_titles = [best_title] if titles[best_title] > 0.7 else []
    return final_titles

In [ ]:
final_titles = get_titles(f"{root}/2026-03-29/SQ-ST303.mp3")
print(f"Songs: {'_'.join(final_titles)}")

In [76]:
for d in ["2026-06-27", "2026-06-20", "2026-06-14", "2026-05-07", "2026-04-11", "2026-03-08", "2026-03-07", "2026-02-14", "2026-02-08", "2026-02-07", "2026-02-01", "2026-01-25"]:
    matches = match_zoom_to_sq(d, verbose=False)
    print(d, json.dumps(matches, indent=2, ensure_ascii=False))

2026-06-27 {
  "ZOOM0429_你的爱_一颗谦卑的心.mp3": "SQ-ST415_你的爱_一颗谦卑的心.mp3",
  "ZOOM0427_将天敞开.mp3": "SQ-ST413_将天敞开.mp3",
  "ZOOM0426_将你最好的献给主.mp3": "SQ-ST412_将你最好的献给主.mp3"
}
2026-06-20 {
  "ZOOM0413_宝贵十架_一生一世.mp3": "SQ-ST398_宝贵十架_一生一世.mp3",
  "ZOOM0414_一生一世.mp3": "SQ-ST399_一生一世.mp3",
  "ZOOM0415_一生一世.mp3": "SQ-ST400_一生一世.mp3",
  "ZOOM0412_耶稣基督是主.mp3": "SQ-ST396_耶稣基督是主.mp3",
  "ZOOM0411_基督精兵前进.mp3": "SQ-ST395_基督精兵前进.mp3",
  "ZOOM0410_伟大的救主.mp3": "SQ-ST393_伟大的救主.mp3"
}
2026-06-14 {
  "ZOOM0405_我們的神_何等恩典.mp3": "SQ-ST389_我們的神_何等恩典.mp3",
  "ZOOM0406_一颗谦卑的心.mp3": "SQ-ST390_一颗谦卑的心.mp3",
  "ZOOM0408_一颗谦卑的心.mp3": "SQ-ST391_一颗谦卑的心.mp3",
  "ZOOM0409_歌颂主爱.mp3": "SQ-ST392_歌颂主爱.mp3"
}
2026-05-07 {}
2026-04-11 {
  "ZOOM0306_你真伟大.mp3": "SQ-ST306_你真伟大.mp3",
  "ZOOM0307_救主耶稣万福恩源.mp3": "SQ-ST307_救主耶稣万福恩源.mp3",
  "ZOOM0312_点燃.mp3": "SQ-ST309_点燃.mp3"
}
2026-03-08 {
  "ZOOM0266_尊贵全能神_无价至宝_我的盼望在于祢.mp3": "SQ-ST268_尊贵全能神_无价至宝_我的盼望在于祢.mp3"
}
2026-03-07 {
  "ZOOM0259_耶稣领我.mp3": "SQ-ST263_耶稣领我.mp3",
  "ZOOM0260_尊贵全能神.mp3